In [ ]:
import time
import json
import os
import sys
import math
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,COMPENSATION_PARA
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import *

from network.layer import Layer

from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
# deviceType参数：0为ReRAM，1为ECRAM
# IsNew32参数：False为v1版本，True为v2版本
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=20,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 10,delay3=50)
chip.add_compiler("../compiler/code/")
chip.compensation.initop("../chip_data/chip6_/pcb_203/")

In [ ]:
root_path = "../chip_data/chip6_/pcb_203/"

# 1. 准备数据

In [ ]:
v,c_expected_from_row,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c_expected_from_row = chip.compensation.compensation_point(r,from_row=True,return_type=0)

v,c_expected_from_col,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
c_expected_from_col = chip.compensation.compensation_point(r,from_row=False,return_type=0)

## 1.0 计算R_out

In [ ]:
crossbar = np.ones((256,256))
v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:
# 注意！！！执行前一定要改名字
compensation = COMPENSATION_PARA()
filename = root_path + "col_r_out_base.npy"
point_read,sum_read = compensation.calculate_r_out_from_r_wire(chip=chip,num=30,from_row=True)
r_out_col = compensation.get_r_out(point_read=point_read,sum_read=sum_read,filename=filename,from_row=True)

In [ ]:
tmp = r_out_col.copy()
for i,v in enumerate(r_out_col):
    if v>25:
        tmp[i]=(tmp[i-1]+tmp[i+1])/2

filename = root_path + "col_r_out.npy"
np.save(filename,tmp)
plt.plot(tmp)
plt.xlabel("col")
plt.ylabel("Ω")
plt.show()

In [ ]:
filename = root_path + "row_r_out_base.npy"
point_read,sum_read = compensation.calculate_r_out_from_r_wire(chip=chip,num=30,from_row=False)
r_out_row = compensation.get_r_out(point_read=point_read,sum_read=sum_read,filename=filename,from_row=False)

In [ ]:
tmp = r_out_row.copy()
for i,v in enumerate(r_out_row):
    if v>25:
        tmp[i]=(tmp[i-1]+tmp[i+1])/2

filename = root_path + "row_r_out.npy"
np.save(filename,tmp)
plt.plot(tmp)
plt.show()
print(r_out_row)

In [ ]:
col_r_out = np.load(root_path + "col_r_out.npy")
plt.plot(col_r_out)
plt.show()

row_r_out = np.load(root_path + "row_r_out.npy")
plt.plot(row_r_out)
plt.show()

## 1.1 从行给信号

In [ ]:
chip.compensation.initop(root_path)
# 10列，两路TIA并行
col_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

for i in range(101):
    row_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(row_index,r,from_row=True,return_type=0,parallel=0)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_row[pos],axis=0)
    real_res[i,:]=r
    real_cond_res[i,:] = cond[col_index]

In [ ]:
startpos,endpos = 0,101
for num in range(10):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

In [ ]:
chip.compensation.initop(root_path)
# 10列，两路TIA并行
col_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

for i in range(101):
    row_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(row_index,r,from_row=True,return_type=0,parallel=16)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_row[pos],axis=0)
    real_res[i,:]=r
    real_cond_res[i,:] = cond[col_index]

In [ ]:
startpos,endpos = 0,101
for num in range(10):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

## 1.2 从列给信号

In [ ]:
chip.compensation.initop(root_path)
# chip.compensation.row_offset=0
# chip.compensation.row_value=None
# chip.compensation.row_r_out=0
# 10列，两路TIA并行
row_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))
for i in range(101):
    col_index = [20+j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=False,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(col_index,r,from_row=False,return_type=0)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_col[pos],axis=1).flatten()
    real_res[i,:]=r
    real_cond_res[i,:] = cond[row_index]

In [ ]:
def compensation_para(input_indexs,offset,value,mult,r_wire,r_out,output_index,actual_output,expected_output,from_row):
    """
        为每一行,每一列,计算补偿的offset
        actual_output,expected_output格式,第一维度为第n次输入(开第n个索引),第二维度为输入的索引号
    """
    num = len(input_indexs)
    ans = np.zeros(num)

    tmp = []
    for k,i in enumerate(input_indexs):
        tmp.append(i)
        min_index,max_index = np.min(tmp),np.max(tmp)
        rw = (255-max_index)*r_wire if (from_row and output_index%2==0) or (not from_row and output_index%2==1) else (min_index)*r_wire

        # 第k次输入，输出索引为output_index,减去r_out,rw
        # offset是需要补偿的部分
        real_r = actual_output[k,output_index]*1e3 -r_out -rw + offset

        if value is not None:
            nums = len(tmp)
            real_r -= r_wire*((max_index-min_index+1)/nums)*(nums**value)
        ans[k]=1/real_r*1e6
    return ans/expected_output[:num,output_index]*mult

In [ ]:
chip.compensation.initop(root_path)
from_row = False
r_wire = chip.compensation.r_wire
r_out = chip.compensation.col_r_out if from_row else chip.compensation.row_r_out
r_offset = 0

x = [i for i in range(20,51)]
y = np.ones(len(x))
offset_fit = np.zeros(256)
for output_index in range(256):
    fun_offset = lambda input_indexs,offset:compensation_para(input_indexs,offset,None,1,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)
    # fun_value = lambda input_indexs,value:compensation_offset(input_indexs,r_offset[output_index],value,1,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)
    # fun_value_mult = lambda input_indexs,value,mult:compensation_offset(input_indexs,r_offset[output_index],value,mult,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)


    params, covariance = curve_fit(fun_offset,x, y, p0=[1.0],bounds=([-30], [30]))
    offset_fit[output_index] = params[0]

np.save(root_path + "row_offset.npy",offset_fit)

In [ ]:
chip.compensation.initop(root_path)
from_row = False
r_wire = chip.compensation.r_wire
r_out = chip.compensation.col_r_out if from_row else chip.compensation.row_r_out
r_offset = np.load(root_path + "row_offset.npy")

x = [i for i in range(20,121)]
y = np.ones(len(x))
value_fit = np.zeros(256)
mult_fit = np.zeros(256)
for output_index in range(256):
    # fun_offset = lambda input_indexs,offset:compensation_offset(input_indexs,offset,None,1,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)
    fun_value = lambda input_indexs,value:compensation_para(input_indexs,r_offset[output_index],value,1,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)
    fun_value_mult = lambda input_indexs,value,mult:compensation_para(input_indexs,r_offset[output_index],value,mult,r_wire,r_out[output_index],output_index,real_res,expected_res,from_row = from_row)


    params, covariance = curve_fit(fun_value,x, y, p0=[0.5],bounds=([0], [1]))
    # params, covariance = curve_fit(fun_value,x, y, p0=[0.5],bounds=([0], [1]))
    value_fit[output_index] = params[0]
np.save(root_path + "row_value.npy",value_fit)

In [ ]:
def compensation_value_from_col(x,mult,row):
    """
        计算补偿
    """
    rows = len(x)
    return real_cond_res[:rows,row]/expected_res[:rows,row]*mult

cnt = 101
ans = np.zeros(256)
for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_col(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]

# print(ans)

In [ ]:
startpos,endpos = 0,101
for num in range(10):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"row={row_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"row={row_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

In [ ]:
chip.compensation.initop(root_path)
# chip.compensation.row_offset=0
# chip.compensation.row_value=0
# chip.compensation.row_r_out=0
# 10列，两路TIA并行
row_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

chip.setting.tia_map = [j for j in range(8) for i in range(2)]

for i in range(101):
    col_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=False,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(col_index,r,from_row=False,return_type=0,parallel=8)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_col[pos],axis=1).flatten()
    real_res[i,:]=r
    real_cond_res[i,:] = cond[row_index]

In [ ]:
startpos,endpos = 0,101
for num in range(9):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("uS")
    plt.xlabel("row nums")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(np.ones(endpos-startpos-1),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.ylabel("*100%")
    plt.xlabel("row nums")
    plt.legend()
    plt.show()

# 2. 计算补偿参数

## 2.1 从行给信号列出

In [ ]:
r_wire = chip.compensation.r_wire
r_out_col = chip.compensation.col_r_out


In [ ]:
def compensation_offset_from_row(x,offset,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 31
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,offset:compensation_offset_from_row(x,offset,i),[i for i in range(cnt)], np.ones(cnt), p0=[0],bounds=([-15], [15]))
    ans[i] = params[0]

In [ ]:
np.save(root_path + "col_offset_parallel.npy",ans)
print(ans)

In [ ]:
offset_col = np.load(root_path + "col_offset_parallel.npy")
def compensation_value_from_row(x,value,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset_col[col]  - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 101
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_row(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]

In [ ]:
np.save(root_path + "col_value_parallel_16.npy",ans)
print(ans)

## 2.1 从列给信号

In [ ]:
chip.compensation.initop(root_path)
r_wire = chip.compensation.r_wire
r_out_row = chip.compensation.row_r_out

print(r_out_row)

In [ ]:
def compensation_offset_from_col(x,offset,row):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if row%2==1 else 0
        r_out = r_out_row[row]
        real_r = real_res[i,row]*1e3 -r_out -sum_rw + offset
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,row]

cnt = 31
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,offset:compensation_offset_from_col(x,offset,i),[i for i in range(cnt)], np.ones(cnt), p0=[0],bounds=([-10], [10]))
    ans[i] = params[0]

In [ ]:
plt.plot(ans)
plt.show()
np.save(root_path + "row_offset.npy",ans)

In [ ]:
offset_row = np.load(root_path + "row_offset.npy")
def compensation_value_from_col(x,value,row):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if row%2==1 else 0
        r_out = r_out_row[row]
        real_r = real_res[i,row]*1e3 -r_out -sum_rw + offset_row[row]  - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,row]

cnt = 101
ans = np.zeros(256)
ans2 = np.zeros(256)
for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_col(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]

In [ ]:
np.save(root_path + "row_value_parallel.npy",ans)
print(ans)

In [ ]:

def compensation_value_from_col(x,offset,value,row):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if row%2==1 else 0
        r_out = r_out_row[row]
        real_r = real_res[i,row]*1e3 -r_out -sum_rw + offset - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,row]

cnt = 81
ans_offset = np.zeros(256)
ans_value = np.zeros(256)
for i in range(256):
    params, covariance = curve_fit(lambda x,offset,value:compensation_value_from_col(x,offset,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0,0.5],bounds=([-100,0], [100,10]))
    ans_offset[i] = params[0]
    ans_value[i] = params[1]

In [ ]:
print(ans_offset)
plt.plot(ans_offset)
plt.show()
plt.plot(ans_value)
print(ans_value)
plt.show()

In [ ]:
np.save(root_path + "row_value_parallel.npy",ans_value)
np.save(root_path + "row_offset.npy",ans_offset)
# print(ans)

# 3.补偿结果

### 3.1 从行给信号列出